In [20]:
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import models , layers
from tensorflow.keras.optimizers import Adam , RMSprop
import tensorflow as tf

## read and clean the data

In [21]:
def cleanData(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

In [22]:
dataset = pd.read_csv('sentiment140.csv')
dataset = dataset[['text' , 'sentiment']]

dataset['text'] = dataset['text'].apply(cleanData)
labelE=LabelEncoder()
dataset['sentiment'] = labelE.fit_transform(dataset['sentiment'])

## tokenize and padding

In [23]:
max_word = 20000
max_len = 100
tokenizer = Tokenizer(num_words=max_word)
tokenizer.fit_on_texts(dataset['text'])

sequences = tokenizer.texts_to_sequences(dataset['text'])
X = pad_sequences(sequences, maxlen=max_len)
y = dataset['sentiment']

x_train , x_test , y_train , y_test = train_test_split(X , y , test_size=0.2, random_state=42)
print(x_train , y_train)

[[   0    0    0 ...    4  157  159]
 [   0    0    0 ...  697   21  112]
 [   0    0    0 ...   14    3  130]
 ...
 [   0    0    0 ...  242   54  291]
 [   0    0    0 ...  209 3464   86]
 [   0    0    0 ... 1927   14  341]] 14307    0
17812    0
11020    0
15158    0
24990    1
        ..
6265     0
11284    1
38158    1
860      0
15795    1
Name: sentiment, Length: 32000, dtype: int64


## load GloVe embeddings

In [24]:
embedding_dimensions = 100
embedding_index = {}

with open('glove.6B.100d.txt' , encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.array(values[1:] , dtype='float32')
        embedding_index[word] = vector

word_index = tokenizer.word_index

embedding_matrix = np.zeros((max_word, embedding_dimensions))

for word , i in word_index.items():
    if i < max_word:
        vector = embedding_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector

In [ ]:
def build_CNN(optimizer = 'adam' , drop_out = 0):
    model = models.Sequential()
    model.add(
        layers.Embedding(
            input_dim=max_word,
            output_dim=embedding_dimensions,
            weights=[embedding_matrix],
            trainable=False
        )
    )
    model.add(layers.Conv1D(128, kernel_size=5, activation='relu'))
    model.add(layers.MaxPooling1D(2))

    model.add(layers.Conv1D(64, kernel_size=5, activation='relu'))
    model.add(layers.GlobalAveragePooling1D())

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(drop_out))

    model.add(layers.Dense(1, activation='sigmoid'))
    if optimizer == 'adam':
        optimizer = Adam()
    elif optimizer == 'rmsprop':
        optimizer = RMSprop()

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
Models_to_train = []
results = {}

x_train , x_val , y_train , y_val = train_test_split(x_train , y_train , test_size=0.1, random_state=42)

tf.config.run_functions_eagerly(True)

model1 = build_CNN(optimizer= 'adam', drop_out=0.2)
model2 = build_CNN(optimizer= 'adam', drop_out=0.5)
model3 = build_CNN(optimizer= 'adam', drop_out=0.7)
model4 = build_CNN(optimizer= 'rmsprop', drop_out=0.2)
model5 = build_CNN(optimizer= 'rmsprop', drop_out=0.5)
Models_to_train.append(model1)
Models_to_train.append(model2)
Models_to_train.append(model3)
Models_to_train.append(model4)
Models_to_train.append(model5)
# print(results)

In [ ]:
for i in range(len(Models_to_train)):
    print(f"-----------------------Model {i+1}training-------------------------")
    Models_to_train[i].fit(x_train, y_train, validation_data=(x_val, y_val), epochs=5, batch_size=32, verbose=1)
    loss , acc = Models_to_train[i].evaluate(x_test, y_test)
    results[Models_to_train[i]] = (loss, acc)

print(results)

-----------------------Model 1training-------------------------
Epoch 1/5


/usr/local/lib/python3.12/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


900/900 ━━━━━━━━━━━━━━━━━━━━ 75s 79ms/step - accuracy: 0.6671 - loss: 0.6041 - val_accuracy: 0.7425 - val_loss: 0.5229
Epoch 2/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 71s 79ms/step - accuracy: 0.7470 - loss: 0.5173 - val_accuracy: 0.7494 - val_loss: 0.5125
Epoch 3/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 69s 77ms/step - accuracy: 0.7784 - loss: 0.4692 - val_accuracy: 0.7531 - val_loss: 0.5163
Epoch 4/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 70s 78ms/step - accuracy: 0.8072 - loss: 0.4249 - val_accuracy: 0.7450 - val_loss: 0.5376
Epoch 5/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 69s 77ms/step - accuracy: 0.8345 - loss: 0.3774 - val_accuracy: 0.7387 - val_loss: 0.5403
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.7399 - loss: 0.5465
-----------------------Model 2training-------------------------
Epoch 1/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 70s 77ms/step - accuracy: 0.6600 - loss: 0.6170 - val_accuracy: 0.7353 - val_loss: 0.5387
Epoch 2/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 70s 78ms/step - accuracy: 0.7394 - loss: 0.5317 - val_acc

In [ ]:
for i in range(len(Models_to_train)):
    print(f'{Models_to_train[i].name}: Loss = {results[Models_to_train[i]][0]}, Accuracy = {results[Models_to_train[i]][1]}')

sequential: Loss = 0.5465430617332458, Accuracy = 0.7398750185966492
sequential_1: Loss = 0.5316386818885803, Accuracy = 0.7422500252723694
sequential_2: Loss = 0.5277315378189087, Accuracy = 0.7418749928474426
sequential_3: Loss = 0.522421658039093, Accuracy = 0.7519999742507935
sequential_4: Loss = 0.5561274886131287, Accuracy = 0.7366250157356262


#### the most efficient model is model number 4 *(optimizer = RMSprop , drop out = 0.2)* with accuracy = 0.76 and loss = 0.51

# AutoEncoder

In [25]:
input_layer = layers.Input(shape=(max_len,))
embedding = layers.Embedding(
    input_dim=max_word,
    output_dim=embedding_dimensions,
    weights=[embedding_matrix],
    input_length=max_len,
    trainable=False
)(input_layer)

#################
#    encoder    #
#################

encoder = layers.Conv1D(128 , 3, activation='relu')(embedding)
encoder = layers.GlobalMaxPool1D()(encoder)


#################
#     repeat    #
#################

repeat = layers.RepeatVector(max_len)(encoder)

#################
#    decoder    #
#################
decoder = layers.Conv1D(128 , 3, activation='relu', padding='same')(repeat)
# Removed GlobalMaxPool1D as it collapses the time dimension
output = layers.TimeDistributed(
    layers.Dense(embedding_dimensions)
)(decoder)


#################
#     model     #
#################
autoEncoder = tf.keras.Model(inputs=input_layer, outputs=output)

autoEncoder.compile(
    optimizer='adam',
    loss = 'mse'
)

autoEncoder.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 100, 100)       │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 98, 128)        │        38,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_2 (RepeatVector)  │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 100, 128)       │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 100, 100)       │        12,900 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,100,708 (8.01 MB)

 Trainable params: 100,708 (393.39 KB)

 Non-trainable params: 2,000,000 (7.63 MB)

In [26]:
X_autoEncoder = X
y_autoEncoder = np.expand_dims(X_autoEncoder , axis=-1)

autoEncoder.fit(X_autoEncoder, y_autoEncoder , epochs=10, batch_size=32 , validation_split=0.1 , verbose=1)



Epoch 1/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 1082368.1250 - val_loss: 702525.8750
Epoch 2/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 1068239.8750 - val_loss: 699780.1875
Epoch 3/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1063160.6250 - val_loss: 697975.3125
Epoch 4/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1060560.6250 - val_loss: 697855.3750
Epoch 5/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 1058737.3750 - val_loss: 697538.3125
Epoch 6/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1056608.8750 - val_loss: 696066.5000
Epoch 7/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 1052469.5000 - val_loss: 695049.7500
Epoch 8/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 1045225.5000 - val_loss: 687976.1875
Epoch 9/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1037194.5625 - val_loss: 685052.8125
Epoch 10/10
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 1030663.1875 - val_loss: 685601.5000


In [27]:
losses_per_sequence = []

batch_size = 64
for i in range(0, len(X_autoEncoder), batch_size):
    batch_X = X_autoEncoder[i:i+batch_size]
    batch_y_embeddings = embedding_matrix[batch_X]

    batch_pred = autoEncoder.predict(batch_X)

    squared_error = tf.square(batch_y_embeddings - batch_pred)

    batch_loss_per_sequence = tf.reduce_mean(squared_error, axis=[1, 2])

    losses_per_sequence.extend(batch_loss_per_sequence.numpy())

losses_np = np.array(losses_per_sequence)

threshold = losses_np.mean() + 2 * losses_np.std()
anomalies = losses_np > threshold

print(f"Number of anomalies found: {anomalies.sum()}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━